# ThinkGuard — Qwen3.5-9B 의미 재라벨링

**준비물:** 이 노트북, `thinkguard_colab_vllm_bundle.zip`, Google Drive, Colab L4 GPU.
위에서부터 차례대로 실행하세요. 첫 표본 검증을 마친 뒤 전체 실행을 켭니다.

- 모델: `RedHatAI/Qwen3.5-9B-FP8-dynamic`, vLLM, non-thinking, 텍스트 전용.
- L4 시작 설정: 배치 8, GPU 메모리 비율 0.85, 최대 길이 7,168, eager 실행.
- 설치가 PyTorch를 변경할 수 있습니다. 셀 3에서 중단되면 세션을 다시 시작하고 셀 1부터 실행하세요.
- 원본: StudyChat 16,851건 + RECIPE4U 1,913건. 기존 주석은 덮어쓰지 않습니다.
- 결과: `MyDrive/ThinkGuard_NLP/qwen35_9b_vllm_fp8_v1/`에 저장합니다.
- 10건마다 해시가 붙은 불변 체크포인트 저장. 강제 종료 시 최대 9건을 다시 처리할 수 있습니다.
- 모델·입력·프롬프트·실행 설정이 바뀌면 다른 실행 이름을 사용해야 합니다.
- 현재 입력 한도 6,000토큰. 넘는 대화는 잘라 읽지 않고 `context_overflow`로 기록합니다.
- `ok`는 JSON·인용 검증 통과라는 뜻입니다. 라벨 정확성·사람 검토 완료를 보증하지 않습니다.
- 원본·학생 발화는 공개 저장소나 공개 노트북 출력에 올리지 말고 개인 Drive에 보관하세요.

GPU 할당·CUDA 호환성과 실제 속도는 연결 후 확인합니다. 이 노트북은 로컬에서
구문·입력·근거 검증·체크포인트 재개를 검사했으며 아직 GPU 추론으로 검증하지 않았습니다.

공식 참고: [모델 카드](https://huggingface.co/Qwen/Qwen3.5-9B),
[vLLM Qwen3.5 공식 레시피](https://github.com/vllm-project/recipes/blob/main/models/Qwen/Qwen3.5-9B.yaml).


## 1. GPU 확인

Colab에서 런타임 유형을 GPU로 설정하세요. GPU가 없으면 즉시 멈춥니다.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경 → GPU로 설정한 뒤 다시 실행하세요.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("VRAM GiB:", round(gpu.total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if gpu.total_memory / 2**30 < 14:
    raise RuntimeError("이 노트북의 9B 시험 설정에는 14 GiB 이상 GPU를 권장합니다. 설정을 먼저 점검하세요.")


## 2. Drive 연결·입력 업로드

함께 제공한 ZIP을 한 번만 업로드합니다. 다음 세션은 Drive에서 읽습니다.


In [ ]:
# ruff: noqa: F821
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ThinkGuard_NLP")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "qwen35_9b_vllm_fp8_v1"  # 같은 설정으로 재개할 때는 바꾸지 마세요.
BUNDLE_NAME = "thinkguard_colab_vllm_bundle.zip"
BUNDLE = PROJECT_DIR / BUNDLE_NAME
WORK = Path("/content/thinkguard_colab_vllm")
WORK.mkdir(exist_ok=True)
if not BUNDLE.exists():
    print(f"제공받은 {BUNDLE_NAME} 하나를 업로드하세요.")
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise ValueError("파일 이름을 확인하세요.")
    BUNDLE.write_bytes(uploaded[BUNDLE_NAME])
    del uploaded
shutil.copyfile(BUNDLE, WORK / "bundle.zip")
with zipfile.ZipFile(WORK / "bundle.zip") as archive:
    if archive.testzip() is not None:
        raise ValueError("ZIP checksum error")
    for name in archive.namelist():
        if Path(name).name != name or name.startswith("."):
            raise ValueError("Unexpected archive path")
    archive.extractall(WORK)
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
for name, expected in manifest["files"].items():
    with (WORK / name).open("rb") as stream:
        observed = hashlib.file_digest(stream, "sha256").hexdigest()
    if observed != expected:
        raise ValueError(f"업로드 파일 검증 실패: {name}")
sys.path.insert(0, str(WORK))
print("원본 행 수:", manifest["counts"], "진단 표본:", manifest["references"])
print("Drive 저장 위치:", PROJECT_DIR)


## 3. 추론 패키지 설치

첫 설치 버전을 Drive에 기록하고 재접속 시 같은 버전 설치를 시도합니다.


In [ ]:
# ruff: noqa: F821
import importlib.metadata
import subprocess
import sys

packages = ["vllm", "torch", "transformers", "safetensors", "sentencepiece", "huggingface-hub"]
environment_file = PROJECT_DIR / "colab_vllm_package_versions.json"

def versions():
    result = {}
    for name in packages:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

before = versions()
if environment_file.exists():
    previous = json.loads(environment_file.read_text(encoding="utf-8"))
    requirements = [f"{name}=={previous[name]}" for name in packages]
else:
    # Let the vLLM wheel select its matching PyTorch/CUDA dependencies.
    requirements = ["vllm>=0.17.0", "sentencepiece", "huggingface-hub"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *requirements])
installed = {name: importlib.metadata.version(name) for name in packages}
environment_file.write_text(json.dumps(installed, indent=2), encoding="utf-8")
print(installed)
restart_marker = WORK / "restart_required.json"
if before != installed:
    restart_marker.write_text(json.dumps({"pid": __import__("os").getpid()}), encoding="utf-8")
if restart_marker.exists():
    if json.loads(restart_marker.read_text())["pid"] == __import__("os").getpid():
        raise RuntimeError("설치 완료. 런타임 → 세션 다시 시작 후 셀 1부터 다시 실행하세요. 지금 모델을 로드하지 마세요.")
    restart_marker.unlink()
print("vLLM 설치 확인 완료. 다음 셀을 실행하세요.")


## 4. vLLM Qwen3.5-9B FP8 로드

첫 실행에서 공식 모델 SHA를 고정합니다. 모델 다운로드에는 시간이 걸립니다.


In [ ]:
# ruff: noqa: F821
from huggingface_hub import HfApi
from inference_vllm import Annotator, load_model
from journal import atomic_json
from runner import configuration

MODEL_ID = "RedHatAI/Qwen3.5-9B-FP8-dynamic"
if (WORK / "restart_required.json").exists():
    raise RuntimeError("설치 후 세션을 다시 시작하고 셀 1부터 재실행하세요.")
if "model" in globals():
    raise RuntimeError("모델이 이미 로드되어 있습니다. 다시 로드하려면 세션을 다시 시작하세요.")
revision_file = PROJECT_DIR / f"{RUN_NAME}_model_revision.json"
if revision_file.exists():
    model_pin = json.loads(revision_file.read_text(encoding="utf-8"))
    if model_pin["model_id"] != MODEL_ID:
        raise ValueError("Model ID changed; use a new RUN_NAME")
else:
    model_pin = {"model_id": MODEL_ID, "revision": HfApi().model_info(MODEL_ID).sha}
    atomic_json(revision_file, model_pin)
print("고정 모델:", model_pin)
model, tokenizer, model_metadata = load_model(MODEL_ID, model_pin["revision"])
annotator = Annotator(model, tokenizer, max_input_tokens=6000, max_new_tokens=768)
run_config = configuration(manifest, model_metadata, annotator)
run_root = PROJECT_DIR / RUN_NAME
run_root.mkdir(exist_ok=True)
atomic_json(run_root / "current_hardware.json", model_metadata)
print("추론 엔진:", model_metadata["engine"], "배치 크기:", annotator.batch_size)
print("GPU 메모리 사용 비율:", model_metadata["engine_settings"]["gpu_memory_utilization"])
print("로드 완료. 먼저 다음 셀의 표본 검증을 실행하세요.")


## 5. 20건 표본 검증

자료별 10건. 결과·근거·속도를 확인합니다. 비교 라벨은 AI 초안이며 사람 정답이 아닙니다.


In [ ]:
# ruff: noqa: F821
import random

from runner import diagnostic, export_review, read_inputs, run

pilot_inputs = list(read_inputs(WORK / "pilot_inputs.jsonl.gz"))
references = [json.loads(line) for line in (WORK / "references.jsonl").open(encoding="utf-8")]
rng = random.Random(20261116)
pilot_selection = []
for dataset in ("studychat", "recipe4u"):
    subset = [r for r in pilot_inputs if r["dataset"] == dataset]
    rng.shuffle(subset)
    pilot_selection.extend(subset[:10])
pilot_journal = run(pilot_selection, annotator, run_root / "pilot", run_config)
diagnostic_report = diagnostic(pilot_journal, references)
print(json.dumps(diagnostic_report, ensure_ascii=False, indent=2))
review_path = export_review(pilot_journal, pilot_selection)
print("표본 원문·모델 판단 검토 파일:", review_path)
print("기존 라벨은 AI 초안입니다. 일치율은 독립 정확도가 아닙니다.")


## 6. 전체 라벨링·재개

표본 확인 후 RUN_FULL=True. 동일 RUN_NAME·설정으로 다시 실행하면 저장된 ID를 건너뜁니다.


In [ ]:
# ruff: noqa: F821
from runner import read_inputs, run

# 표본 원문·근거·처리속도를 확인한 뒤 아래 값만 True로 바꾸세요.
RUN_FULL = False
MAX_NEW_ROWS = None  # 전체 대상. 시험 처리량 제한은 정수 입력. 재실행 시 저장된 ID는 건너뜁니다.
if RUN_FULL:
    valid_pilot = sum(r["status"] == "ok" for r in pilot_journal.records.values())
    if valid_pilot != len(pilot_selection):
        raise RuntimeError("표본 중 오류가 있습니다. 원인을 확인한 뒤 전체 처리를 시작하세요.")
    full_journal = run(
        read_inputs(WORK / "inputs.jsonl.gz"), annotator,
        run_root / "full", run_config, limit=MAX_NEW_ROWS,
    )
else:
    print("표본을 확인한 뒤 RUN_FULL=True로 바꾸면 전체 라벨링을 진행합니다.")


## 7. 결과·커버리지 확인

오류·긴 입력·보류는 성공으로 숨기지 않습니다. 의미 판단도 사람 검토 전 AI 초안입니다.


In [ ]:
# ruff: noqa: F821
from journal import Journal, atomic_json

if (run_root / "full/run_config.json").exists():
    saved = Journal(run_root / "full", run_config)
    summary = saved.summary()
    summary["expected"] = manifest["total"]
    summary["remaining_unprocessed"] = manifest["total"] - summary["processed"]
    summary["all_rows_semantically_processed"] = (
        summary["remaining_unprocessed"] == 0
        and summary["statuses"].get("ok", 0) == manifest["total"]
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("전체 결과:", saved.export())
    exceptional = [r for r in saved.records.values() if r["status"] != "ok"]
    uncertain = [r for r in saved.records.values() if r.get("labels", {}).get("uncertain")]
    atomic_json(run_root / "full/errors.json", exceptional)
    atomic_json(run_root / "full/uncertain.json", uncertain)
    atomic_json(run_root / "full/coverage.json", summary)
    print("오류와 보류 결과도 Drive에 따로 저장했습니다.")
else:
    print("전체 실행은 아직 시작되지 않았습니다. 표본 결과는 pilot 폴더에 있습니다.")


## 연결이 끊겼을 때

GPU 런타임을 다시 연결하고 셀 1부터 실행합니다. Drive ZIP과 모델 SHA는 재사용합니다.
모델 캐시는 Colab 임시 디스크여서 새 런타임에서는 다시 내려받을 수 있습니다.
표본과 전체 결과는 Drive에 남습니다. 설정이 같으면 이미 체크포인트된 행은 재처리하지 않습니다.
중단 시 노트북의 정지 버튼은 finally 저장을 시도합니다. 강제 종료에서는 마지막 9건이 유실될 수 있습니다.

`context_overflow`, `oom`, `invalid_output`은 의미 검토 성공이 아니며 errors.json으로 모읍니다.
오류 행은 자동으로 무한 재시도하지 않습니다. 원인 수정 후 별도 실행으로 재검토해야 합니다.
전체 커버리지의 모든 status가 ok여야 모든 행이 의미 추론을 통과한 상태입니다.
라벨 null 보류는 ok인 행에도 남을 수 있으며 별도 사람 검토 대상입니다.
패키지·PyTorch가 달라 재개가 거부되면 기존 결과를 보존하고 새 RUN_NAME으로 시작하세요.
